In [39]:
# 1. setup
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATASET_DIR = os.path.join('..', 'dataset')
OUTPUT_DIR  = os.path.join('..', 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'kaplan_meier'), exist_ok=True)

print('ready')

ready


In [40]:
# 2. load TNBC
tcga_brca = pd.read_csv(os.path.join(DATASET_DIR, 'brca_tcga_clinical_data.tsv'), sep='\t', low_memory=False)

er_col   = 'ER Status By IHC'
pr_col   = 'PR status by ihc'
her2_col = 'IHC-HER2'

tnbc = tcga_brca[
    (tcga_brca[er_col]   == 'Negative') &
    (tcga_brca[pr_col]   == 'Negative') &
    (tcga_brca[her2_col] == 'Negative')
].copy().reset_index(drop=True)

print(f'TNBC: {len(tnbc)}')
tnbc[['Patient ID', er_col, pr_col, her2_col]].head()

TNBC: 117


,Patient ID,ER Status By IHC,PR status by ihc,IHC-HER2
0,TCGA-A1-A0SK,Negative,Negative,Negative
1,TCGA-A1-A0SP,Negative,Negative,Negative
2,TCGA-A2-A04U,Negative,Negative,Negative
3,TCGA-A2-A0CM,Negative,Negative,Negative
4,TCGA-A2-A0D0,Negative,Negative,Negative


In [41]:
# 3. build scores
import tarfile
from bioconverge.utils import parse_gmt

RNASEQ_TAR = os.path.join(DATASET_DIR, 'gdac.broadinstitute.org_BRCA.Merge_rnaseqv2__illuminahiseq_rnaseqv2__unc_edu__Level_3__RSEM_genes_normalized__data.Level_3.2016012800.0.0.tar.gz')
MAF_TAR    = os.path.join(DATASET_DIR, 'gdac.broadinstitute.org_BRCA.Mutation_Packager_Oncotated_Calls.Level_3.2016012800.0.0.tar.gz')
GMT_PATH   = os.path.join(DATASET_DIR, 'h.all.v2023.2.Hs.symbols.gmt')

hallmarks    = parse_gmt(GMT_PATH)
IMMUNE_GENES = ['CD8A', 'CD8B', 'GZMA', 'PRF1']
PROLIF_GENES = ['MKI67']
EMT_GENES    = hallmarks.get('HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION', [])
print(f'EMT genes: {len(EMT_GENES)}')

def safe_norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else pd.Series(0.5, index=s.index)

def load_rnaseq(tar_path, patient_ids, gene_sets):
    with tarfile.open(tar_path) as t:
        member = next(m for m in t.getmembers() if m.name.endswith('.data.txt'))
        raw = pd.read_csv(t.extractfile(member), sep='\t', header=0, index_col=0, skiprows=[1])
    raw.columns = [c[:12] for c in raw.columns]
    raw.index   = [g.split('|')[0] for g in raw.index]
    raw = raw[~raw.index.duplicated(keep='first')]
    pid_set = set(patient_ids)
    common_cols = list(dict.fromkeys(c for c in raw.columns if c in pid_set))
    rna = raw[common_cols].T
    rna.index.name = 'patient_id'
    rna = rna.groupby('patient_id').mean().T
    unique_pids = list(dict.fromkeys(p for p in patient_ids if p in rna.columns))
    rna = rna[unique_pids]
    scores = {}
    for name, genes in gene_sets.items():
        avail = [g for g in genes if g in rna.index]
        scores[name] = rna.loc[avail].mean(axis=0) if avail else pd.Series(np.nan, index=unique_pids)
    df = pd.DataFrame(scores)
    df.index.name = 'patient_id'
    return df.reset_index(), unique_pids

def load_tmb(tar_path, patient_ids):
    pid_set = set(patient_ids)
    counts = {p: 0 for p in pid_set}
    with tarfile.open(tar_path) as t:
        for member in t.getmembers():
            if not member.name.endswith('.maf.txt'):
                continue
            pid = os.path.basename(member.name)[:12]
            if pid not in pid_set:
                continue
            try:
                df = pd.read_csv(t.extractfile(member), sep='\t', comment='#',
                                 usecols=['Hugo_Symbol'], low_memory=False)
                counts[pid] = len(df)
            except Exception:
                pass
    return pd.Series(counts, name='mutation_raw')

patient_ids = list(dict.fromkeys(tnbc['Patient ID'].tolist()))
print(f'patients: {len(patient_ids)}')

rna_df = None
rna_matched = []
if os.path.exists(RNASEQ_TAR):
    print('loading RNA-seq')
    rna_df, rna_matched = load_rnaseq(RNASEQ_TAR, patient_ids,
                                       {'prolif_raw': PROLIF_GENES,
                                        'immune_raw': IMMUNE_GENES,
                                        'emt_raw':    EMT_GENES})
    print(f'RNA-seq matched: {len(rna_matched)}')
else:
    print('RNA-seq not found, using proxies')

print('loading mutations')
tmb = load_tmb(MAF_TAR, patient_ids)
print(f'MAF matched: {(tmb > 0).sum()}')

clin = tnbc[['Patient ID', 'Fraction Genome Altered', 'Longest Dimension']].copy()
clin.columns = ['patient_id', 'fga', 'longest_dim']
clin['fga']         = pd.to_numeric(clin['fga'], errors='coerce')
clin['longest_dim'] = pd.to_numeric(clin['longest_dim'], errors='coerce')
clin = clin.drop_duplicates(subset='patient_id').reset_index(drop=True)

if rna_df is not None:
    score_df = clin.merge(rna_df, on='patient_id', how='left')
    score_df['proliferation_score'] = safe_norm(np.log1p(score_df['prolif_raw'].fillna(score_df['prolif_raw'].median())))
    score_df['immune_score']        = safe_norm(np.log1p(score_df['immune_raw'].fillna(score_df['immune_raw'].median())))
    score_df['emt_score']           = safe_norm(np.log1p(score_df['emt_raw'].fillna(score_df['emt_raw'].median())))
else:
    score_df = clin.copy()
    rng = np.random.default_rng(42)
    score_df['proliferation_score'] = safe_norm(pd.Series(rng.uniform(0, 1, len(score_df))))
    score_df['immune_score']        = safe_norm(pd.Series(rng.uniform(0, 1, len(score_df))))
    score_df['emt_score']           = safe_norm(pd.Series(rng.uniform(0, 1, len(score_df))))

score_df['mutation_raw']   = score_df['patient_id'].map(tmb.to_dict()).fillna(0)
score_df['genomic_score']  = safe_norm(score_df['fga'].fillna(score_df['fga'].median()))
score_df['size_score']     = safe_norm(score_df['longest_dim'].fillna(score_df['longest_dim'].median()))
score_df['mutation_score'] = safe_norm(np.log1p(score_df['mutation_raw']))

score_df = score_df[['patient_id', 'genomic_score', 'size_score',
                      'proliferation_score', 'immune_score', 'mutation_score', 'emt_score']].dropna().reset_index(drop=True)

score_df.to_csv(os.path.join(OUTPUT_DIR, 'score_matrix.csv'), index=False)
print(f'score matrix: {score_df.shape}')
score_df.head()

EMT genes: 200
patients: 116
loading RNA-seq
RNA-seq matched: 115
loading mutations
MAF matched: 103
score matrix: (116, 7)


,patient_id,genomic_score,size_score,proliferation_score,immune_score,mutation_score,emt_score
0,TCGA-A1-A0SK,0.439924,0.5,0.945098,0.150112,0.704357,0.072899
1,TCGA-A1-A0SP,0.414034,0.5,0.727943,0.267557,0.525654,0.285542
2,TCGA-A2-A04U,0.705590,0.5,0.731760,0.146914,0.551991,0.333755
3,TCGA-A2-A0CM,0.415090,0.5,0.858545,0.669108,0.547114,0.376783
4,TCGA-A2-A0D0,0.746169,0.5,0.845508,0.571168,0.574059,0.304036


In [42]:
# 4. layer1 concordance
from bioconverge.layer1 import ConcordanceAnalyzer

l1 = ConcordanceAnalyzer(score_df, 'patient_id')
l1.fit(n_archetypes=3, n_bootstrap=500)

conc = l1.concordance()
conv = l1.convergence()
arch = l1.archetypes()
stab = l1.stability()

conc.to_csv(os.path.join(OUTPUT_DIR, 'concordance.csv'), index=False)
conv.to_csv(os.path.join(OUTPUT_DIR, 'convergence.csv'), index=False)
arch.to_csv(os.path.join(OUTPUT_DIR, 'archetypes.csv'), index=False)

print(conc.to_string())
print(f'stability: {stab}')

fitting concordance
layer1 done
                score_a              score_b  spearman_rho     ci_lo     ci_hi  n_patients
0         genomic_score           size_score           NaN       NaN       NaN         116
1         genomic_score  proliferation_score      0.196110 -0.004043  0.366560         116
2         genomic_score         immune_score     -0.416873 -0.560370 -0.236940         116
3         genomic_score       mutation_score      0.210398  0.020947  0.408535         116
4         genomic_score            emt_score     -0.094101 -0.296938  0.094878         116
5            size_score  proliferation_score           NaN       NaN       NaN         116
6            size_score         immune_score           NaN       NaN       NaN         116
7            size_score       mutation_score           NaN       NaN       NaN         116
8            size_score            emt_score           NaN       NaN       NaN         116
9   proliferation_score         immune_score      0.033870

In [43]:
# 5. concordance plot
score_names = sorted(set(conc['score_a'].tolist() + conc['score_b'].tolist()))
n = len(score_names)
mat = np.zeros((n, n))
idx_map = {s: i for i, s in enumerate(score_names)}
for _, row in conc.iterrows():
    i, j = idx_map[row['score_a']], idx_map[row['score_b']]
    v = row['spearman_rho'] if not np.isnan(row['spearman_rho']) else 0
    mat[i, j] = v
    mat[j, i] = v
np.fill_diagonal(mat, 1.0)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(mat, cmap='RdBu', vmin=-1, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(score_names, rotation=45, ha='right')
ax.set_yticklabels(score_names)
plt.colorbar(im, ax=ax, label='spearman rho')
ax.set_title('concordance matrix')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'concordance_matrix.png'), dpi=150)
plt.show()
print('saved')

saved


In [44]:
# 6. convergence plot
colors = {'convergent_high': '#2ecc71', 'convergent_mid': '#f39c12', 'convergent_low': '#e74c3c', 'unknown': '#95a5a6'}
fig, ax = plt.subplots(figsize=(8, 4))
for stratum, grp in conv.groupby('stratum'):
    ax.hist(grp['convergence_index'], bins=15, alpha=0.7, label=stratum, color=colors.get(stratum, 'gray'))
ax.set_xlabel('convergence index')
ax.set_ylabel('count')
ax.set_title('patient convergence strata')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'convergence_strata.png'), dpi=150)
plt.show()
print(conv['stratum'].value_counts().to_string())

stratum
convergent_mid     40
convergent_low     38
convergent_high    38


In [45]:
# 7. archetype summary
arch_summary = arch.merge(conv, on='patient_id')
print(arch_summary.groupby('archetype')['stratum'].value_counts().unstack(fill_value=0).to_string())
disc = l1.discordance()
print(f'discordant: {len(disc)}')

stratum    convergent_high  convergent_low  convergent_mid
archetype                                                 
0                       23              17              15
1                        9              16              20
2                        6               5               5
discordant: 38


In [46]:
# 8. score metadata
from bioconverge.layer3 import HypothesisGenerator
from bioconverge.layer4 import LEHMANN_SIGNATURES

score_metadata = {
    'genomic_score':       {'process': 'DNA repair instability',
                            'modality': 'genomic',
                            'genes': ['BRCA1', 'BRCA2', 'ATM', 'TP53', 'CHEK2']},
    'size_score':          {'process': 'tumor growth and growth factor signaling',
                            'modality': 'imaging',
                            'genes': ['EGFR', 'IGF1R', 'MET', 'ERBB2', 'FGFR1']},
    'proliferation_score': {'process': 'cell cycle proliferation',
                            'modality': 'transcriptomic',
                            'genes': ['MKI67', 'PCNA', 'TOP2A', 'CCNB1', 'CDK1']},
    'immune_score':        {'process': 'immune activation',
                            'modality': 'transcriptomic',
                            'genes': ['CD8A', 'CD8B', 'GZMA', 'PRF1', 'IFNG']},
    'mutation_score':      {'process': 'mutational burden',
                            'modality': 'genomic',
                            'genes': ['TP53', 'PIK3CA', 'PTEN', 'RB1', 'CDH1']},
    'emt_score':           {'process': 'epithelial-mesenchymal transition EMT',
                            'modality': 'transcriptomic',
                            'genes': ['VIM', 'CDH2', 'FN1', 'TWIST1', 'SNAI1']},
}

for subtype, keywords in LEHMANN_SIGNATURES.items():
    print(f'{subtype}: {keywords}')
print()
for sc, meta in score_metadata.items():
    proc = meta['process']
    hits = [s for s, kws in LEHMANN_SIGNATURES.items() if any(kw.lower() in proc.lower() for kw in kws)]
    print(f'{sc}: {proc!r} -> {hits if hits else "no match"}')

BL1: ['DNA damage response', 'BRCA', 'cell cycle', 'checkpoint']
BL2: ['growth factor signaling', 'IGF1R', 'EGFR', 'PI3K']
M: ['epithelial-mesenchymal transition', 'EMT', 'TGF', 'WNT']
IM: ['immune activation', 'interferon', 'cytokine', 'T cell']

genomic_score: 'DNA repair instability' -> no match
size_score: 'tumor growth and growth factor signaling' -> ['BL2']
proliferation_score: 'cell cycle proliferation' -> ['BL1']
immune_score: 'immune activation' -> ['IM']
mutation_score: 'mutational burden' -> no match
emt_score: 'epithelial-mesenchymal transition EMT' -> ['M']


In [47]:
# 9. hypothesis generation
l3 = HypothesisGenerator(score_metadata, arch)
l3.generate()

hyp = l3.hypotheses()
hyp.to_csv(os.path.join(OUTPUT_DIR, 'hypotheses_raw.csv'), index=False)
print(f'hypotheses: {len(hyp)}')
hyp[['archetype', 'process', 'db_support', 'reactome_pathway', 'pubmed_count', 'pubmed_flag']].head(10)

generating hypotheses
hypotheses done
hypotheses: 18


,archetype,process,db_support,reactome_pathway,pubmed_count,pubmed_flag
0,0,DNA repair instability,2,,1047,convergent_with_prior_knowledge
1,0,tumor growth and growth factor signaling,2,,14373,convergent_with_prior_knowledge
2,0,cell cycle proliferation,2,,14814,convergent_with_prior_knowledge
3,0,immune activation,2,,13280,convergent_with_prior_knowledge
4,0,mutational burden,2,,1811,convergent_with_prior_knowledge
5,0,epithelial-mesenchymal transition EMT,2,,5516,convergent_with_prior_knowledge
6,1,DNA repair instability,2,,1047,convergent_with_prior_knowledge
7,1,tumor growth and growth factor signaling,2,,14373,convergent_with_prior_knowledge
8,1,cell cycle proliferation,2,,14814,convergent_with_prior_knowledge
9,1,immune activation,2,,13280,convergent_with_prior_knowledge


In [48]:
# 10. cross support
cs = l3.cross_support()
cs.to_csv(os.path.join(OUTPUT_DIR, 'cross_support.csv'), index=False)
print(cs.to_string())

   archetype  mean_support  max_support  n_hypotheses
0          0           2.0            2             6
1          1           2.0            2             6
2          2           2.0            2             6


In [49]:
# 11. find validation files
from bioconverge.layer4 import ValidationEngine

clinical_tar       = None
mut_tar            = None
metabric_path      = None
tcga_brca_path_val = None

for fname in os.listdir(DATASET_DIR):
    fpath = os.path.join(DATASET_DIR, fname)
    if 'clinical' in fname and fname.endswith('.tar.gz'):
        clinical_tar = fpath
    elif fname == 'clinical.tsv':
        clinical_tar = fpath
    elif 'Mutation_Packager' in fname and fname.endswith('.tar.gz'):
        mut_tar = fpath
    elif 'metabric' in fname and fname.endswith('.tsv'):
        metabric_path = fpath
    elif 'brca_tcga' in fname and fname.endswith('.tsv'):
        tcga_brca_path_val = fpath

print(f'clinical: {clinical_tar}')
print(f'mutations: {mut_tar}')
print(f'metabric: {metabric_path}')
print(f'tcga brca: {tcga_brca_path_val}')

clinical: ..\dataset\clinical.project-tcga-brca.2026-05-24.tar.gz
mutations: ..\dataset\gdac.broadinstitute.org_BRCA.Mutation_Packager_Oncotated_Calls.Level_3.2016012800.0.0.tar.gz
metabric: ..\dataset\brca_metabric_clinical_data.tsv
tcga brca: ..\dataset\brca_tcga_clinical_data.tsv


In [50]:
# 12. run validation
l4 = ValidationEngine(
    clinical_tar_path=clinical_tar,
    mut_tar_path=mut_tar,
    metabric_path=metabric_path,
    tcga_brca_path=tcga_brca_path_val,
    dataset_dir=DATASET_DIR,
)

km_dir = os.path.join(OUTPUT_DIR, 'kaplan_meier')
l4.validate(
    hypotheses_df=hyp,
    archetypes_df=arch,
    score_df=score_df,
    patient_col='patient_id',
    score_metadata=score_metadata,
    km_output_dir=km_dir,
)

replication_rate = l4.compute_replication_rate(hyp)
print(f'replication rate: {replication_rate}')

validation start
survival analysis
loading clinical
loading mutations
km 0 saved
km 1 saved
km 2 saved
metabric replication
loading metabric
fitting concordance
layer1 done
generating hypotheses
hypotheses done
benchmark validation
loading tcga brca
fitting concordance
layer1 done
generating hypotheses
hypotheses done
fitting concordance
layer1 done
generating hypotheses
hypotheses done
fitting concordance
layer1 done
generating hypotheses
hypotheses done
fitting concordance
layer1 done
generating hypotheses
hypotheses done
literature scoring
validation done
replication rate: 0.5


In [51]:
# diagnostic
surv = l4.survival_results()
meta = l4.metabric_results()
bench = l4.benchmark_results()
lit = l4.literature_results()

print("survival archetypes with p<0.05:")
if surv and not surv.get('skipped'):
    print(surv['results'][['archetype','logrank_pvalue','survival_signal']].to_string())
else:
    print(surv)

print("\nmetabric replicated processes:")
if meta and not meta.get('skipped'):
    print(f"rate: {meta.get('replication_rate')}")
    print(f"replicated: {meta.get('replicated_processes')}")
else:
    print(meta)

print("\nbenchmark per subtype:")
if bench and not bench.get('skipped'):
    for k,v in bench['lehmann_results'].items():
        print(f"{k}: precision={v['precision']:.2f} recall={v['recall']:.2f}")
    print(f"process_bench_pass: {bench.get('process_bench_pass')}")
else:
    print(bench)

print("\nliterature support:")
if lit and not lit.get('skipped'):
    print(lit['results'][['process','pubmed_count','literature_support']].to_string())
else:
    print(lit)

survival archetypes with p<0.05:
   archetype  logrank_pvalue  survival_signal
0          0        0.402023            False
1          1        0.367216            False
2          2        0.984745            False

metabric replicated processes:
rate: 0.5
replicated: {'tumor growth and growth factor signaling', 'cell cycle proliferation', 'mutational burden'}

benchmark per subtype:
BL1: precision=0.17 recall=0.25
BL2: precision=0.17 recall=0.25
M: precision=0.17 recall=0.25
IM: precision=0.17 recall=0.25
process_bench_pass: {'dna damage response', 'checkpoint', 'immune activation', 't cell', 'epithelial-mesenchymal transition', 'emt', 'interferon', 'wnt', 'cell cycle', 'brca', 'pi3k', 'igf1r', 'growth factor signaling', 'tgf', 'egfr', 'cytokine'}

literature support:
                                     process  pubmed_count  literature_support
0                     DNA repair instability          1047                True
1   tumor growth and growth factor signaling         14373  

In [52]:
# 13. survival results
surv = l4.survival_results()
if surv and not surv.get('skipped'):
    print(f'matched: {surv["n_matched"]}')
    surv['results'].to_csv(os.path.join(OUTPUT_DIR, 'survival_results.csv'), index=False)
    print(surv['results'].to_string())
    for fname in os.listdir(km_dir):
        if fname.endswith('.png'):
            img = plt.imread(os.path.join(km_dir, fname))
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(fname.replace('.png', ''))
            plt.tight_layout()
            plt.show()
else:
    print(f'survival skipped: {surv}')

matched: 111
   archetype  n_archetype  n_rest  logrank_pvalue  survival_signal  n_tp53_mutant_archetype  n_tp53_mutant_rest
0          0           55      56        0.402023            False                       42                  29
1          1           40      71        0.367216            False                       29                  42
2          2           16      95        0.984745            False                        0                  71


In [53]:
# 14. benchmark results
bench = l4.benchmark_results()
if bench and not bench.get('skipped'):
    print(f'mean precision: {bench["mean_precision"]:.3f}')
    print(f'mean recall: {bench["mean_recall"]:.3f}')
    for subtype, vals in bench['lehmann_results'].items():
        print(f'{subtype}: precision={vals["precision"]:.2f}, recall={vals["recall"]:.2f}')
else:
    print(f'benchmark skipped: {bench}')

mean precision: 0.167
mean recall: 0.250
BL1: precision=0.17, recall=0.25
BL2: precision=0.17, recall=0.25
M: precision=0.17, recall=0.25
IM: precision=0.17, recall=0.25


In [54]:
# 15. tier breakdown
tiered = l4.tiered_hypotheses()
tiered.to_csv(os.path.join(OUTPUT_DIR, 'hypotheses_tiered.csv'), index=False)

print(tiered['confidence_tier'].value_counts().to_string())

tier_a = tiered[tiered['confidence_tier'] == 'A']
tier_b = tiered[tiered['confidence_tier'] == 'B']
tier_c = tiered[tiered['confidence_tier'] == 'C']
print(f'Tier A: {len(tier_a)}, Tier B: {len(tier_b)}, Tier C: {len(tier_c)}')

tier_counts = tiered['confidence_tier'].value_counts().reindex(['A', 'B', 'C'], fill_value=0)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(tier_counts.index, tier_counts.values, color=['#27ae60', '#f39c12', '#e74c3c'])
ax.set_xlabel('tier')
ax.set_ylabel('hypotheses')
ax.set_title('tier A/B/C breakdown')
for bar, v in zip(bars, tier_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, str(v), ha='center', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'tier_breakdown.png'), dpi=150)
plt.show()
print('saved')

confidence_tier
B    9
A    6
C    3
Tier A: 6, Tier B: 9, Tier C: 3
saved


In [55]:
# 16. best hypothesis
show_tier = tier_a if len(tier_a) > 0 else tier_b if len(tier_b) > 0 else tier_c
best = show_tier.sort_values(['validation_score', 'db_support'], ascending=False).iloc[0]
for col in ['archetype', 'process', 'modality', 'hypothesis', 'db_support',
            'reactome_pathway', 'enrichr_term', 'pubmed_count', 'pubmed_flag', 'validation_score', 'confidence_tier']:
    if col in best.index:
        print(f'{col}: {best[col]}')

archetype: 0
process: tumor growth and growth factor signaling
modality: imaging
hypothesis: Archetype 0 shows tumor growth and growth factor signaling (imaging) signal. Top pathway: . Top gene set: .
db_support: 2
reactome_pathway: 
enrichr_term: 
pubmed_count: 14373
pubmed_flag: convergent_with_prior_knowledge
validation_score: 3
confidence_tier: A


In [56]:
# 17. discordance analysis
conv_low = conv[conv['stratum'] == 'convergent_low']['patient_id'].tolist()
print(f'discordant: {len(conv_low)}')

arch_merged = arch.merge(conv, on='patient_id')
arch_merged.to_csv(os.path.join(OUTPUT_DIR, 'archetype_convergence.csv'), index=False)
for a in sorted(arch_merged['archetype'].unique()):
    sub = arch_merged[arch_merged['archetype'] == a]
    disc_sub = sub[sub['stratum'] == 'convergent_low']
    print(f'archetype {a}: {len(sub)} patients, {len(disc_sub)} discordant ({100*len(disc_sub)/max(len(sub),1):.0f}%)')

discordant: 38
archetype 0: 55 patients, 17 discordant (31%)
archetype 1: 45 patients, 16 discordant (36%)
archetype 2: 16 patients, 5 discordant (31%)


In [57]:
# 18. full report
from bioconverge.report import BioConvergeResult

result = BioConvergeResult(l1=l1, l2=None, l3=l3, l4=l4, skipped=['layer2'])
result.report(OUTPUT_DIR)

for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        print(f'{f}: {os.path.getsize(fpath)} bytes')

saved per_patient_scores
saved hypotheses
saved repro log
saved summary.html
report done
archetype_convergence.csv: 5965 bytes
archetypes.csv: 1878 bytes
benchmark_comparison.csv: 280 bytes
benchmark_comparison.png: 68637 bytes
benchmark_replication.csv: 146 bytes
benchmark_replication.png: 42504 bytes
concordance.csv: 1166 bytes
concordance_matrix.png: 64672 bytes
convergence.csv: 5723 bytes
convergence_index_clipped.png: 36002 bytes
convergence_index_properties.png: 173145 bytes
convergence_strata.png: 30970 bytes
cross_support.csv: 82 bytes
discordant_subgroup_enrichr.csv: 1026 bytes
discordant_subgroup_mw.csv: 185 bytes
discordant_subgroup_pathways.png: 41138 bytes
discordant_subgroup_scores.png: 83849 bytes
hypotheses_ranked.csv: 4474 bytes
hypotheses_raw.csv: 4369 bytes
hypotheses_tiered.csv: 4474 bytes
km_discordant_subgroup.png: 36972 bytes
paper3_gwas_connection.csv: 17 bytes
per_patient_scores.csv: 5965 bytes
reproducibility_log.txt: 6228 bytes
score_matrix.csv: 12964 bytes
s

In [58]:
# 19. summary
print('done')
print(f'patients: {len(score_df)}')
print(f'archetypes: {len(arch["archetype"].unique())}')
print(f'hypotheses: {len(hyp)}')
print(f'Tier A: {(tiered["confidence_tier"]=="A").sum()}')
print(f'Tier B: {(tiered["confidence_tier"]=="B").sum()}')
print(f'Tier C: {(tiered["confidence_tier"]=="C").sum()}')
print(f'output: {OUTPUT_DIR}')

done
patients: 116
archetypes: 3
hypotheses: 18
Tier A: 6
Tier B: 9
Tier C: 3
output: ..\output


In [59]:
# 20. sensitivity k archetypes
from bioconverge.layer1 import ConcordanceAnalyzer
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = os.path.join('..', 'output')

score_cols = ['genomic_score', 'size_score', 'proliferation_score',
              'immune_score', 'mutation_score', 'emt_score']

k_results = []
for k in [2, 3, 4]:
    ca = ConcordanceAnalyzer(score_df, 'patient_id')
    ca.fit(n_archetypes=k, n_bootstrap=500)
    stab = ca.stability()
    k_results.append({'k': k, 'mean_ari': stab['mean_ari'], 'std_ari': stab['std_ari']})
    print(f'k={k} ARI={stab["mean_ari"]:.3f} std={stab["std_ari"]:.3f}')

k_df = pd.DataFrame(k_results)
k_df.to_csv(os.path.join(OUTPUT_DIR, 'sensitivity_k.csv'), index=False)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(k_df['k'].astype(str), k_df['mean_ari'], yerr=k_df['std_ari'],
       color='steelblue', capsize=5)
ax.axhline(0.6, color='red', linestyle='dashed', label='threshold 0.6')
ax.set_xlabel('k archetypes')
ax.set_ylabel('mean ARI')
ax.set_title('archetype stability across k')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sensitivity_k.png'), dpi=150)
plt.show()
print('saved')

fitting concordance
layer1 done
k=2 ARI=0.321 std=0.306
fitting concordance
layer1 done
k=3 ARI=0.399 std=0.194
fitting concordance
layer1 done
k=4 ARI=0.447 std=0.158
saved


In [60]:
# 21. bootstrap convergence
bootstrap_sizes = [100, 200, 300, 400, 500]
boot_results = []
for n in bootstrap_sizes:
    ca = ConcordanceAnalyzer(score_df, 'patient_id')
    ca.fit(n_archetypes=3, n_bootstrap=n)
    stab = ca.stability()
    boot_results.append({'n_bootstrap': n, 'mean_ari': stab['mean_ari']})
    print(f'n={n} ARI={stab["mean_ari"]:.3f}')

boot_df = pd.DataFrame(boot_results)
boot_df.to_csv(os.path.join(OUTPUT_DIR, 'sensitivity_bootstrap.csv'), index=False)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(boot_df['n_bootstrap'], boot_df['mean_ari'], marker='o', color='steelblue')
ax.set_xlabel('bootstrap samples')
ax.set_ylabel('mean ARI')
ax.set_title('bootstrap convergence')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sensitivity_bootstrap.png'), dpi=150)
plt.show()
print('saved')

fitting concordance
layer1 done
n=100 ARI=0.405
fitting concordance
layer1 done
n=200 ARI=0.427
fitting concordance
layer1 done
n=300 ARI=0.413
fitting concordance
layer1 done
n=400 ARI=0.421
fitting concordance
layer1 done
n=500 ARI=0.400
saved


In [61]:
# 22. spearman vs pearson
from scipy.stats import spearmanr, pearsonr

spearman_rhos = []
pearson_rhos  = []
pair_labels   = []

for i in range(len(score_cols)):
    for j in range(i + 1, len(score_cols)):
        x = score_df[score_cols[i]].values
        y = score_df[score_cols[j]].values
        mask = ~(np.isnan(x) | np.isnan(y))
        if mask.sum() < 5:
            continue
        sr, _ = spearmanr(x[mask], y[mask])
        pr, _ = pearsonr(x[mask], y[mask])
        spearman_rhos.append(float(sr))
        pearson_rhos.append(float(pr))
        pair_labels.append(f'{score_cols[i][:8]} vs {score_cols[j][:8]}')

corr_df = pd.DataFrame({
    'pair': pair_labels,
    'spearman': spearman_rhos,
    'pearson': pearson_rhos
})
corr_df.to_csv(os.path.join(OUTPUT_DIR, 'sensitivity_spearman_pearson.csv'), index=False)

sp_arr = np.array(spearman_rhos)
pe_arr = np.array(pearson_rhos)

valid = ~(np.isnan(sp_arr) | np.isnan(pe_arr))
sp_clean = sp_arr[valid]
pe_clean = pe_arr[valid]

print(f'valid pairs: {valid.sum()} of {len(sp_arr)}')

if len(set(sp_clean)) > 1 and len(set(pe_clean)) > 1:
    rank_corr, _ = spearmanr(sp_clean, pe_clean)
    print(f'rank correlation: {rank_corr:.3f}')
else:
    rank_corr = None
    print('not enough variation')

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(sp_clean, pe_clean, color='steelblue')
lims = [min(sp_clean.tolist() + pe_clean.tolist()) - 0.05,
        max(sp_clean.tolist() + pe_clean.tolist()) + 0.05]
ax.plot(lims, lims, color='red', linestyle='dashed', label='identity')
ax.set_xlabel('Spearman rho')
ax.set_ylabel('Pearson r')
label = f'rank r={rank_corr:.2f}' if rank_corr is not None else 'low variation'
ax.set_title(f'Spearman vs Pearson ({label})')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sensitivity_spearman_pearson.png'), dpi=150)
plt.show()
print('saved')

valid pairs: 10 of 15
rank correlation: 0.952
saved


In [62]:
print("spearman values:", [round(x,3) for x in spearman_rhos])
print("pearson values:", [round(x,3) for x in pearson_rhos])
print("unique spearman:", len(set([round(x,3) for x in spearman_rhos])))

spearman values: [nan, 0.196, -0.417, 0.21, -0.094, nan, nan, nan, nan, 0.034, 0.171, -0.201, -0.083, -0.163, -0.156]
pearson values: [nan, 0.244, -0.43, 0.102, -0.116, nan, nan, nan, nan, 0.051, 0.166, -0.167, -0.041, -0.167, -0.141]
unique spearman: 15


In [63]:
# 23. score removal robustness
from sklearn.metrics import adjusted_rand_score

base_ca = ConcordanceAnalyzer(score_df, 'patient_id')
base_ca.fit(n_archetypes=3, n_bootstrap=200)
base_labels = base_ca.archetypes().set_index('patient_id')['archetype']

removal_results = []
for drop_col in score_cols:
    remaining = [c for c in score_cols if c != drop_col]
    sub_df = score_df[['patient_id'] + remaining].copy()
    ca_sub = ConcordanceAnalyzer(sub_df, 'patient_id')
    ca_sub.fit(n_archetypes=3, n_bootstrap=200)
    sub_labels = ca_sub.archetypes().set_index('patient_id')['archetype']
    common = base_labels.index.intersection(sub_labels.index)
    ari = adjusted_rand_score(base_labels[common], sub_labels[common])
    removal_results.append({'dropped': drop_col, 'ari_vs_full': ari})
    print(f'drop {drop_col}: ARI={ari:.3f}')

rem_df = pd.DataFrame(removal_results)
rem_df.to_csv(os.path.join(OUTPUT_DIR, 'sensitivity_score_removal.csv'), index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(len(rem_df)), rem_df['ari_vs_full'], color='steelblue')
ax.axhline(0.6, color='red', linestyle='dashed', label='threshold 0.6')
ax.set_xticks(range(len(rem_df)))
ax.set_xticklabels([c[:12] for c in rem_df['dropped']], rotation=30, ha='right')
ax.set_ylabel('ARI vs full model')
ax.set_title('archetype stability when one score is removed')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sensitivity_score_removal.png'), dpi=150)
plt.show()
print('saved')

fitting concordance
layer1 done
fitting concordance
layer1 done
drop genomic_score: ARI=0.469
fitting concordance
layer1 done
drop size_score: ARI=1.000
fitting concordance
layer1 done
drop proliferation_score: ARI=0.664
fitting concordance
layer1 done
drop immune_score: ARI=0.219
fitting concordance
layer1 done
drop mutation_score: ARI=0.419
fitting concordance
layer1 done
drop emt_score: ARI=0.896
saved


In [64]:
# 24. tier reproducibility
from sklearn.metrics import adjusted_rand_score

base_labels = arch['archetype'].values

seed_results = []
for seed in range(10):
    ca_run = ConcordanceAnalyzer(score_df, 'patient_id')
    ca_run.fit(n_archetypes=3, n_bootstrap=200, random_state=seed)
    run_labels = ca_run.archetypes()['archetype'].values
    ari = adjusted_rand_score(base_labels, run_labels)
    seed_results.append({'seed': seed, 'ari_vs_seed0': ari})
    print(f'seed={seed} ARI vs base={ari:.3f}')

seed_df = pd.DataFrame(seed_results)
seed_df.to_csv(os.path.join(OUTPUT_DIR, 'sensitivity_tier_reproducibility.csv'), index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(seed_df['seed'].astype(str), seed_df['ari_vs_seed0'], color='steelblue')
ax.axhline(0.6, color='red', linestyle='dashed', label='threshold 0.6')
ax.set_xlabel('random seed')
ax.set_ylabel('ARI vs base clustering')
ax.set_title('archetype stability across 10 seeds')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sensitivity_tier_reproducibility.png'), dpi=150)
plt.show()
print('saved')

fitting concordance
layer1 done
seed=0 ARI vs base=0.533
fitting concordance
layer1 done
seed=1 ARI vs base=1.000
fitting concordance
layer1 done
seed=2 ARI vs base=0.533
fitting concordance
layer1 done
seed=3 ARI vs base=0.160
fitting concordance
layer1 done
seed=4 ARI vs base=0.679
fitting concordance
layer1 done
seed=5 ARI vs base=0.664
fitting concordance
layer1 done
seed=6 ARI vs base=0.978
fitting concordance
layer1 done
seed=7 ARI vs base=0.533
fitting concordance
layer1 done
seed=8 ARI vs base=0.664
fitting concordance
layer1 done
seed=9 ARI vs base=0.938
saved


In [65]:
# 25. paper3 connection GWAS test
from bioconverge.utils import api_get

GWAS_URL = "https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search"
GWAS_ASSOC_URL = "https://www.ebi.ac.uk/gwas/rest/api/singleNucleotidePolymorphisms/search/findByGene"

paper3_queries = [
    "cohesin DNA repair",
    "meiosis cancer",
    "STAG3 colorectal",
    "structural variant colorectal cancer",
    "7q22 colorectal",
]

paper3_results = []
for query in paper3_queries:
    try:
        r = api_get(GWAS_URL, params={"query": query, "page": 0, "size": 5})
        traits = r.json().get("_embedded", {}).get("efoTraits", [])
        for t in traits[:3]:
            paper3_results.append({
                "query": query,
                "trait": t.get("trait", ""),
                "uri": t.get("uri", ""),
            })
            print(f'query: {query!r} -> trait: {t.get("trait","")}')
        if not traits:
            print(f'query: {query!r} -> no results')
    except Exception as e:
        print(f'query: {query!r} -> fail: {e}')

p3_df = pd.DataFrame(paper3_results) if paper3_results else pd.DataFrame(
    columns=["query", "trait", "uri"])
p3_df.to_csv(os.path.join(OUTPUT_DIR, 'paper3_gwas_connection.csv'), index=False)

colorectal_hits = p3_df[p3_df['trait'].str.lower().str.contains('colorectal|colon|bowel', na=False)]
dna_repair_hits = p3_df[p3_df['trait'].str.lower().str.contains('dna|repair|cohesin|meiosis', na=False)]

print(f'\ncolorectal hits: {len(colorectal_hits)}')
print(f'DNA repair hits: {len(dna_repair_hits)}')

if len(colorectal_hits) > 0 or len(dna_repair_hits) > 0:
    print('GWAS Catalog recovered paper3 relevant signals')
else:
    print('no direct hits this query, API may need broader terms')

print('\npaper3 connection saved')

query: 'cohesin DNA repair' -> no results
query: 'meiosis cancer' -> no results
query: 'STAG3 colorectal' -> no results
query: 'structural variant colorectal cancer' -> no results
query: '7q22 colorectal' -> no results

colorectal hits: 0
DNA repair hits: 0
no direct hits this query, API may need broader terms

paper3 connection saved


In [66]:
# 26. benchmarking vs baselines
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import NMF, PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from bioconverge.layer1 import ConcordanceAnalyzer
import os

OUTPUT_DIR = os.path.join('..', 'output')

score_cols = ['genomic_score', 'size_score', 'proliferation_score',
              'immune_score', 'mutation_score', 'emt_score']

X_raw = score_df[score_cols].values
X_filled = X_raw.copy()
for j in range(X_filled.shape[1]):
    col = X_filled[:, j]
    med = np.nanmedian(col)
    col[np.isnan(col)] = med if not np.isnan(med) else 0.0

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled)

# bioconverge base labels already computed in cell 4
base_labels = arch['archetype'].values

def bootstrap_ari(labels, X, n_iters=200, random_state=42):
    rng = np.random.default_rng(random_state)
    aris = []
    for _ in range(n_iters):
        idx = rng.integers(0, len(X), size=len(X))
        try:
            km = KMeans(n_clusters=3, random_state=None, n_init=3, max_iter=100)
            km.fit(X[idx])
            pred = km.predict(X)
            aris.append(adjusted_rand_score(labels, pred))
        except Exception:
            pass
    return float(np.mean(aris)) if aris else np.nan, float(np.std(aris)) if aris else np.nan

def lehmann_recovery(labels, score_df, score_cols):
    lehmann_map = {
        'immune_score':        'IM',
        'proliferation_score': 'BL1',
        'emt_score':           'M',
        'genomic_score':       'BL1',
        'mutation_score':      'BL1',
        'size_score':          'BL2',
    }
    recovered = 0
    for arch_id in np.unique(labels):
        mask = labels == arch_id
        means = score_df[score_cols][mask].mean()
        top_score = means.idxmax()
        expected_subtype = lehmann_map.get(top_score, None)
        if expected_subtype is not None:
            recovered += 1
    return recovered / len(np.unique(labels))

results = []

# bioconverge layer1
print('bioconverge')
bc_ari, bc_std = bootstrap_ari(base_labels, X_scaled)
bc_sil = silhouette_score(X_scaled, base_labels)
bc_leh = lehmann_recovery(base_labels, score_df, score_cols)
results.append({
    'method': 'bioconverge (Layer 1)',
    'mean_ari': bc_ari,
    'std_ari': bc_std,
    'silhouette': bc_sil,
    'lehmann_recovery': bc_leh,
})

# PCA + KMeans baseline
print('PCA baseline')
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)
km_pca = KMeans(n_clusters=3, random_state=42, n_init=10)
pca_labels = km_pca.fit_predict(X_pca)
pca_ari, pca_std = bootstrap_ari(pca_labels, X_pca)
pca_sil = silhouette_score(X_scaled, pca_labels)
pca_leh = lehmann_recovery(pca_labels, score_df, score_cols)
results.append({
    'method': 'PCA + KMeans',
    'mean_ari': pca_ari,
    'std_ari': pca_std,
    'silhouette': pca_sil,
    'lehmann_recovery': pca_leh,
})

# NMF baseline
print('NMF baseline')
X_pos = X_filled - X_filled.min(axis=0)
nmf = NMF(n_components=3, random_state=42, max_iter=500)
X_nmf = nmf.fit_transform(X_pos)
km_nmf = KMeans(n_clusters=3, random_state=42, n_init=10)
nmf_labels = km_nmf.fit_predict(X_nmf)
nmf_ari, nmf_std = bootstrap_ari(nmf_labels, X_nmf)
nmf_sil = silhouette_score(X_scaled, nmf_labels)
nmf_leh = lehmann_recovery(nmf_labels, score_df, score_cols)
results.append({
    'method': 'NMF + KMeans',
    'mean_ari': nmf_ari,
    'std_ari': nmf_std,
    'silhouette': nmf_sil,
    'lehmann_recovery': nmf_leh,
})

# Hierarchical clustering
print('hierarchical')
hc = AgglomerativeClustering(n_clusters=3, linkage='ward')
hc_labels = hc.fit_predict(X_scaled)
hc_ari, hc_std = bootstrap_ari(hc_labels, X_scaled)
hc_sil = silhouette_score(X_scaled, hc_labels)
hc_leh = lehmann_recovery(hc_labels, score_df, score_cols)
results.append({
    'method': 'Hierarchical (Ward)',
    'mean_ari': hc_ari,
    'std_ari': hc_std,
    'silhouette': hc_sil,
    'lehmann_recovery': hc_leh,
})

# Gaussian Mixture Model
print('GMM')
gmm = GaussianMixture(n_components=3, random_state=42, n_init=5)
gmm_labels = gmm.fit_predict(X_scaled)
gmm_ari, gmm_std = bootstrap_ari(gmm_labels, X_scaled)
gmm_sil = silhouette_score(X_scaled, gmm_labels)
gmm_leh = lehmann_recovery(gmm_labels, score_df, score_cols)
results.append({
    'method': 'Gaussian Mixture',
    'mean_ari': gmm_ari,
    'std_ari': gmm_std,
    'silhouette': gmm_sil,
    'lehmann_recovery': gmm_leh,
})

# MOFA+ if available
try:
    from mofapy2.run.entry_point import entry_point
    print('MOFA+')
    ent = entry_point()
    data = [[X_filled]]
    ent.set_data_options(scale_groups=False, scale_views=False)
    ent.set_data_matrix(data, likelihoods=['gaussian'],
                        views_names=['scores'], groups_names=['TNBC'])
    ent.set_model_options(factors=3)
    ent.set_train_options(iter=500, seed=42, verbose=False)
    ent.build()
    ent.run()
    Z = ent.model.nodes['Z'].getExpectation()
    if isinstance(Z, list):
        Z = Z[0]
    if Z.ndim == 1:
        Z = Z.reshape(-1, 1)
    print('Z shape:', Z.shape)
    km_mofa = KMeans(n_clusters=3, random_state=42, n_init=10)
    mofa_labels = km_mofa.fit_predict(Z)
    mofa_ari, mofa_std = bootstrap_ari(mofa_labels, Z)
    mofa_sil = silhouette_score(X_scaled, mofa_labels)
    mofa_leh = lehmann_recovery(mofa_labels, score_df, score_cols)
    results.append({
        'method': 'MOFA+',
        'mean_ari': mofa_ari,
        'std_ari': mofa_std,
        'silhouette': mofa_sil,
        'lehmann_recovery': mofa_leh,
    })
    print('MOFA+ done')
except ImportError:
    print('mofapy2 not installed, run: pip install mofapy2')
except Exception as e:
    print(f'MOFA+ failed: {e}')

bench_df = pd.DataFrame(results)
bench_df = bench_df.round(3)
bench_df.to_csv(os.path.join(OUTPUT_DIR, 'benchmark_comparison.csv'), index=False)
print(bench_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
methods = bench_df['method'].tolist()
x = range(len(methods))

axes[0].bar(x, bench_df['mean_ari'], yerr=bench_df['std_ari'],
            color=['steelblue' if 'bioconverge' in m else 'lightgray' for m in methods],
            capsize=4)
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods, rotation=30, ha='right', fontsize=8)
axes[0].set_ylabel('mean ARI')
axes[0].set_title('cluster stability')

axes[1].bar(x, bench_df['silhouette'],
            color=['steelblue' if 'bioconverge' in m else 'lightgray' for m in methods])
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, rotation=30, ha='right', fontsize=8)
axes[1].set_ylabel('silhouette score')
axes[1].set_title('cluster quality')

axes[2].bar(x, bench_df['lehmann_recovery'],
            color=['steelblue' if 'bioconverge' in m else 'lightgray' for m in methods])
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods, rotation=30, ha='right', fontsize=8)
axes[2].set_ylabel('recovery rate')
axes[2].set_title('Lehmann subtype recovery')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'benchmark_comparison.png'), dpi=150)
plt.show()
print('saved')

bioconverge
PCA baseline
NMF baseline
hierarchical
GMM
MOFA+

        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


Features names not provided, using default naming convention:
- feature1_view1, featureD_viewM

Samples names not provided, using default naming convention:
- sample1_group1, sample2_group1, sample1_group2, ..., sampleN_groupG

Successfully loaded view='scores' group='TNBC' with N=116 samples and D=6 features...



Model opti

In [67]:
# 27. replication rate comparison
from bioconverge.layer3 import HypothesisGenerator

replication_results = []

method_labels_map = {
    'bioconverge (Layer 1)': base_labels,
    'PCA + KMeans': pca_labels,
    'NMF + KMeans': nmf_labels,
    'Hierarchical (Ward)': hc_labels,
    'Gaussian Mixture': gmm_labels,
    'MOFA+': mofa_labels,
}

for method_name, labels in method_labels_map.items():
    print(f'replication {method_name}')
    arch_df = pd.DataFrame({'patient_id': score_df['patient_id'].values, 'archetype': labels})
    l3_run = HypothesisGenerator(score_metadata, arch_df)
    l3_run.generate()
    hyp_run = l3_run.hypotheses()
    from bioconverge.layer4 import ValidationEngine
    ve = ValidationEngine(metabric_path=metabric_path)
    ve._run_metabric(score_metadata)
    rate = ve.compute_replication_rate(hyp_run)
    replication_results.append({'method': method_name, 'replication_rate': rate if rate is not None else 0.0})

rep_df = pd.DataFrame(replication_results)
rep_df.to_csv(os.path.join(OUTPUT_DIR, 'benchmark_replication.csv'), index=False)
print(rep_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if 'bioconverge' in m else 'lightgray' for m in rep_df['method']]
ax.bar(range(len(rep_df)), rep_df['replication_rate'], color=colors)
ax.set_xticks(range(len(rep_df)))
ax.set_xticklabels(rep_df['method'], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('replication rate')
ax.set_title('METABRIC replication rate by method')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'benchmark_replication.png'), dpi=150)
plt.show()
print('saved')

replication bioconverge (Layer 1)
generating hypotheses
hypotheses done
metabric replication
loading metabric
fitting concordance
layer1 done
generating hypotheses
hypotheses done
replication PCA + KMeans
generating hypotheses
hypotheses done
metabric replication
loading metabric
fitting concordance
layer1 done
generating hypotheses
hypotheses done
replication NMF + KMeans
generating hypotheses
hypotheses done
metabric replication
loading metabric
fitting concordance
layer1 done
generating hypotheses
hypotheses done
replication Hierarchical (Ward)
generating hypotheses
hypotheses done
metabric replication
loading metabric
fitting concordance
layer1 done
generating hypotheses
hypotheses done
replication Gaussian Mixture
generating hypotheses
hypotheses done
metabric replication
loading metabric
fitting concordance
layer1 done
generating hypotheses
hypotheses done
replication MOFA+
generating hypotheses
hypotheses done
metabric replication
loading metabric
fitting concordance
layer1 done

In [68]:
# 28. discordant subgroup analysis
from scipy.stats import mannwhitneyu

disc_patients = set(conv[conv['stratum'] == 'convergent_low']['patient_id'].tolist())
arch0_patients = set(arch[arch['archetype'] == 0]['patient_id'].tolist())

disc_arch0 = list(disc_patients & arch0_patients)
rest = score_df[~score_df['patient_id'].isin(disc_arch0)]['patient_id'].tolist()

print(f'discordant archetype 0: {len(disc_arch0)}')
print(f'rest: {len(rest)}')

special_scores = score_df[score_df['patient_id'].isin(disc_arch0)][score_cols]
rest_scores = score_df[score_df['patient_id'].isin(rest)][score_cols]

print('\nscore means discordant vs rest:')
for col in score_cols:
    s_mean = special_scores[col].mean()
    r_mean = rest_scores[col].mean()
    print(f'{col}: discordant={s_mean:.3f} rest={r_mean:.3f}')

mw_results = []
for col in score_cols:
    a = special_scores[col].dropna().values
    b = rest_scores[col].dropna().values
    if len(a) < 3 or len(b) < 3:
        mw_results.append({'score': col, 'pvalue': np.nan, 'significant': False})
        continue
    stat, pval = mannwhitneyu(a, b, alternative='two-sided')
    mw_results.append({'score': col, 'pvalue': round(pval, 4), 'significant': pval < 0.05})

mw_df = pd.DataFrame(mw_results)
mw_df.to_csv(os.path.join(OUTPUT_DIR, 'discordant_subgroup_mw.csv'), index=False)
print(mw_df.to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(score_cols):
    axes[i].boxplot([
        special_scores[col].dropna().values,
        rest_scores[col].dropna().values
    ], labels=['discordant\narch0', 'rest'])
    axes[i].set_title(col.replace('_', ' '))
    axes[i].set_ylabel('score')
    pval = mw_df[mw_df['score'] == col]['pvalue'].values[0]
    axes[i].set_xlabel(f'p={pval}')
plt.suptitle('discordant archetype 0 vs rest', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'discordant_subgroup_scores.png'), dpi=150)
plt.show()
print('saved')

discordant archetype 0: 17
rest: 99

score means discordant vs rest:
genomic_score: discordant=0.212 rest=0.480
size_score: discordant=0.500 rest=0.500
proliferation_score: discordant=0.723 rest=0.737
immune_score: discordant=0.614 rest=0.431
mutation_score: discordant=0.567 rest=0.522
emt_score: discordant=0.403 rest=0.374
              score  pvalue  significant
      genomic_score  0.0000         True
         size_score  1.0000        False
proliferation_score  0.2369        False
       immune_score  0.0011         True
     mutation_score  0.3837        False
          emt_score  0.6395        False
saved


In [69]:
# 29. discordant subgroup survival
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from bioconverge.utils import load_clinical_tcga

clinical = load_clinical_tcga(clinical_tar)
merged_surv = score_df[['patient_id']].copy()
merged_surv['group'] = merged_surv['patient_id'].apply(
    lambda x: 'discordant subgroup' if x in disc_arch0 else 'rest')
merged_surv = merged_surv.merge(clinical, on='patient_id', how='inner')

disc_mask = merged_surv['group'] == 'discordant subgroup'
rest_mask = merged_surv['group'] == 'rest'

print(f'matched survival: discordant={disc_mask.sum()} rest={rest_mask.sum()}')

if disc_mask.sum() >= 3 and rest_mask.sum() >= 3:
    res = logrank_test(
        merged_surv.loc[disc_mask, 'OS_days'],
        merged_surv.loc[rest_mask, 'OS_days'],
        merged_surv.loc[disc_mask, 'OS_event'],
        merged_surv.loc[rest_mask, 'OS_event'],
    )
    print(f'logrank p={res.p_value:.4f}')

    fig, ax = plt.subplots(figsize=(8, 5))
    for grp, label, color in [('discordant subgroup', 'discordant arch0', 'red'),
                                ('rest', 'rest', 'steelblue')]:
        mask = merged_surv['group'] == grp
        kmf = KaplanMeierFitter()
        kmf.fit(merged_surv.loc[mask, 'OS_days'],
                merged_surv.loc[mask, 'OS_event'],
                label=f'{label} (n={mask.sum()})')
        kmf.plot_survival_function(ax=ax, color=color)
    ax.set_title(f'discordant subgroup vs rest (p={res.p_value:.4f})')
    ax.set_xlabel('days')
    ax.set_ylabel('survival')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'km_discordant_subgroup.png'), dpi=150)
    plt.show()
    print('saved')
else:
    print('too few patients for survival analysis')

loading clinical
matched survival: discordant=17 rest=94
logrank p=0.2561
saved


In [70]:
# 30. discordant subgroup pathway enrichment using local GMT
from bioconverge.utils import parse_gmt

hallmarks = parse_gmt(GMT_PATH)

disc_gene_set = set(['CD8A', 'CD8B', 'GZMA', 'PRF1', 'IFNG',
                     'BRCA1', 'BRCA2', 'ATM', 'TP53', 'CHEK2',
                     'MKI67', 'PCNA', 'TOP2A', 'CCNB1', 'CDK1'])

from scipy.stats import hypergeom

total_genes = 20000
results = []
for pathway_name, genes in hallmarks.items():
    pathway_set = set(genes)
    overlap = disc_gene_set & pathway_set
    if len(overlap) == 0:
        continue
    M = total_genes
    n = len(pathway_set)
    N = len(disc_gene_set)
    k = len(overlap)
    pval = hypergeom.sf(k - 1, M, n, N)
    results.append({
        'pathway': pathway_name,
        'overlap': k,
        'pathway_size': n,
        'overlap_genes': ', '.join(sorted(overlap)),
        'pvalue': round(pval, 6),
    })

enr_df = pd.DataFrame(results).sort_values('pvalue').reset_index(drop=True)
enr_df.to_csv(os.path.join(OUTPUT_DIR, 'discordant_subgroup_enrichr.csv'), index=False)
print(enr_df.head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
top = enr_df.head(8)
ax.barh(range(len(top)), -np.log10(top['pvalue'] + 1e-10), color='steelblue')
ax.set_yticks(range(len(top)))
ax.set_yticklabels([p.replace('HALLMARK_', '') for p in top['pathway']], fontsize=8)
ax.set_xlabel('-log10(pvalue)')
ax.set_title('hallmark pathway enrichment in discordant subgroup signature genes')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'discordant_subgroup_pathways.png'), dpi=150)
plt.show()
print('saved')

                            pathway  overlap  pathway_size                                       overlap_genes   pvalue
       HALLMARK_ALLOGRAFT_REJECTION        6           200                 BRCA1, CD8A, CD8B, GZMA, IFNG, PRF1 0.000000
               HALLMARK_E2F_TARGETS        8           200 BRCA1, BRCA2, CDK1, CHEK2, MKI67, PCNA, TOP2A, TP53 0.000000
            HALLMARK_G2M_CHECKPOINT        4           200                           BRCA2, CDK1, MKI67, TOP2A 0.000012
                 HALLMARK_APOPTOSIS        3           161                                  BRCA1, PRF1, TOP2A 0.000217
           HALLMARK_MITOTIC_SPINDLE        3           199                                  BRCA2, CDK1, TOP2A 0.000404
                HALLMARK_DNA_REPAIR        2           150                                          PCNA, TP53 0.005503
               HALLMARK_P53_PATHWAY        2           200                                          PCNA, TP53 0.009590
HALLMARK_WNT_BETA_CATENIN_SIGNALING     

In [71]:
# 31. convergence index properties
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = os.path.join('..', 'output')

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# toy example 1: perfectly convergent patient (all scores same direction)
z_conv = np.array([0.8, 0.9, 0.7, 0.85, 0.75])
mu = np.mean(z_conv)
sigma = np.std(z_conv)
c_conv = 1.0 - sigma / (abs(mu) + 1e-8)
axes[0, 0].bar(range(len(z_conv)), z_conv, color='steelblue')
axes[0, 0].set_title(f'convergent patient\nc={c_conv:.3f}')
axes[0, 0].set_xlabel('score index')
axes[0, 0].set_ylabel('z-score')
axes[0, 0].axhline(0, color='black', linewidth=0.5)

# toy example 2: perfectly discordant patient (scores cancel out)
z_disc = np.array([0.9, -0.9, 0.8, -0.8, 0.1])
mu = np.mean(z_disc)
sigma = np.std(z_disc)
c_disc = 1.0 - sigma / (abs(mu) + 1e-8)
axes[0, 1].bar(range(len(z_disc)), z_disc,
               color=['steelblue' if v > 0 else 'red' for v in z_disc])
axes[0, 1].set_title(f'discordant patient\nc={c_disc:.3f}')
axes[0, 1].set_xlabel('score index')
axes[0, 1].set_ylabel('z-score')
axes[0, 1].axhline(0, color='black', linewidth=0.5)

# toy example 3: neutral patient (all near zero)
z_neut = np.array([0.05, -0.03, 0.02, -0.01, 0.04])
mu = np.mean(z_neut)
sigma = np.std(z_neut)
c_neut = 1.0 - sigma / (abs(mu) + 1e-8)
axes[0, 2].bar(range(len(z_neut)), z_neut,
               color=['steelblue' if v > 0 else 'red' for v in z_neut])
axes[0, 2].set_title(f'neutral patient\nc={c_neut:.3f}')
axes[0, 2].set_xlabel('score index')
axes[0, 2].set_ylabel('z-score')
axes[0, 2].axhline(0, color='black', linewidth=0.5)

# property 1: convergence vs std/mean ratio
ratios = np.linspace(0, 5, 200)
c_values = 1.0 - ratios / (1.0 + 1e-8)
axes[1, 0].plot(ratios, c_values, color='steelblue')
axes[1, 0].axhline(0, color='red', linestyle='dashed', label='c=0 boundary')
axes[1, 0].set_xlabel('sigma / |mu|')
axes[1, 0].set_ylabel('convergence index c')
axes[1, 0].set_title('c as function of sigma/|mu|')
axes[1, 0].legend()

# property 2: sensitivity to number of scores
n_scores_range = range(2, 11)
c_stable = []
c_unstable = []
rng = np.random.default_rng(42)
for n in n_scores_range:
    z_s = rng.uniform(0.5, 1.0, n)
    mu_s, sig_s = np.mean(z_s), np.std(z_s)
    c_stable.append(1.0 - sig_s / (abs(mu_s) + 1e-8))
    z_u = rng.uniform(-1.0, 1.0, n)
    mu_u, sig_u = np.mean(z_u), np.std(z_u)
    c_unstable.append(1.0 - sig_u / (abs(mu_u) + 1e-8))
axes[1, 1].plot(list(n_scores_range), c_stable, marker='o', label='convergent pattern', color='steelblue')
axes[1, 1].plot(list(n_scores_range), c_unstable, marker='s', label='discordant pattern', color='red')
axes[1, 1].set_xlabel('number of scores')
axes[1, 1].set_ylabel('convergence index c')
axes[1, 1].set_title('c vs number of scores')
axes[1, 1].legend()

# property 3: comparison to entropy-based measure
from scipy.stats import entropy as scipy_entropy
sigmas = np.linspace(0.01, 2.0, 100)
mu_fixed = 0.5
c_formula = 1.0 - sigmas / (mu_fixed + 1e-8)
axes[1, 2].plot(sigmas, c_formula, color='steelblue', label='bioconverge c')
axes[1, 2].axhline(0, color='red', linestyle='dashed')
axes[1, 2].set_xlabel('sigma (fixed mu=0.5)')
axes[1, 2].set_ylabel('convergence index c')
axes[1, 2].set_title('c decreases linearly with sigma')
axes[1, 2].legend()

plt.suptitle('convergence index properties', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'convergence_index_properties.png'), dpi=150)
plt.show()

print(f'convergent patient c = {c_conv:.3f}')
print(f'discordant patient c = {c_disc:.3f}')
print(f'neutral patient c = {c_neut:.3f}')
print('saved')

convergent patient c = 0.912
discordant patient c = -37.131
neutral patient c = -1.148
saved


In [72]:
# 32. convergence index bounded version
def convergence_bounded(z_scores, clip=True):
    mu = np.mean(z_scores)
    sigma = np.std(z_scores)
    c = 1.0 - sigma / (abs(mu) + 1e-8)
    if clip:
        c = float(np.clip(c, -1.0, 1.0))
    return c

# test same examples
z_conv  = np.array([0.8, 0.9, 0.7, 0.85, 0.75])
z_disc  = np.array([0.9, -0.9, 0.8, -0.8, 0.1])
z_neut  = np.array([0.05, -0.03, 0.02, -0.01, 0.04])

print(f'convergent  unclipped={convergence_bounded(z_conv, clip=False):.3f}  clipped={convergence_bounded(z_conv):.3f}')
print(f'discordant  unclipped={convergence_bounded(z_disc, clip=False):.3f}  clipped={convergence_bounded(z_disc):.3f}')
print(f'neutral     unclipped={convergence_bounded(z_neut, clip=False):.3f}  clipped={convergence_bounded(z_neut):.3f}')

# recompute convergence index on real data with clipping
patient_conv_clipped = []
X_arr = score_df[score_cols].values
global_means = np.nanmean(X_arr, axis=0)
global_stds = np.nanstd(X_arr, axis=0)
global_stds[global_stds == 0] = 1.0

for p_idx in range(len(X_arr)):
    vals = X_arr[p_idx]
    valid = ~np.isnan(vals)
    if valid.sum() < 2:
        patient_conv_clipped.append(np.nan)
        continue
    z = (vals[valid] - global_means[valid]) / global_stds[valid]
    c = convergence_bounded(z, clip=True)
    patient_conv_clipped.append(c)

conv_clipped = pd.Series(patient_conv_clipped)
print(f'\nclipped convergence index range: {conv_clipped.min():.3f} to {conv_clipped.max():.3f}')
print(f'mean: {conv_clipped.mean():.3f}  std: {conv_clipped.std():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(conv['convergence_index'], bins=20, color='lightcoral', alpha=0.7, label='unclipped')
axes[0].set_title('unclipped convergence index')
axes[0].set_xlabel('c value')
axes[0].set_ylabel('count')

axes[1].hist(conv_clipped, bins=20, color='steelblue', alpha=0.7, label='clipped')
axes[1].set_title('clipped convergence index')
axes[1].set_xlabel('c value')
axes[1].set_ylabel('count')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'convergence_index_clipped.png'), dpi=150)
plt.show()
print('saved')

convergent  unclipped=0.912  clipped=0.912
discordant  unclipped=-37.131  clipped=-1.000
neutral     unclipped=-1.148  clipped=-1.000

clipped convergence index range: -1.000 to 0.417
mean: -0.800  std: 0.364
saved
